# 01 — Data Exploration

Exploratory analysis of the synthetic dashboard design dataset.

Covers:
- Dataset statistics (split sizes, field distribution)
- Industry and audience distribution
- KPI frequency analysis
- Chart type distribution in ground-truth recommendations
- Token length distribution (relevant for sequence packing)

In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.helpers import load_jsonl

DATA_DIR = PROJECT_ROOT / 'data'
print(f'Data directory: {DATA_DIR}')

In [ ]:
# Load all splits
train = load_jsonl(str(DATA_DIR / 'train.jsonl'))
val   = load_jsonl(str(DATA_DIR / 'val.jsonl'))
test  = load_jsonl(str(DATA_DIR / 'test.jsonl'))

print(f'Train: {len(train)} examples')
print(f'Val:   {len(val)} examples')
print(f'Test:  {len(test)} examples')
print(f'Total: {len(train) + len(val) + len(test)} examples')

In [ ]:
# Industry distribution
all_data = train + val + test
industry_counts = Counter(item['brief']['industry'] for item in all_data)

print('Industry distribution:')
for industry, count in sorted(industry_counts.items(), key=lambda x: -x[1]):
    print(f'  {industry:<35} {count:3d} ({100*count/len(all_data):.1f}%)')

In [ ]:
# Chart type distribution in ground-truth recommendations
chart_counts = Counter()
for item in all_data:
    for entry in item['recommendation'].get('kpi_task_chart_mapping', []):
        chart = entry.get('recommended_chart', '')
        if chart:
            chart_counts[chart] += 1

print('Chart type distribution (ground truth):')
for chart, count in chart_counts.most_common(20):
    print(f'  {chart:<30} {count:4d}')

In [ ]:
# Token length distribution (approximate, based on word count)
from utils.helpers import format_instruction_prompt

word_counts = []
for item in train:
    prompt = format_instruction_prompt(item['brief'])
    word_counts.append(len(prompt.split()))

import statistics
print(f'Prompt word counts (train):')
print(f'  min   : {min(word_counts)}')
print(f'  mean  : {statistics.mean(word_counts):.0f}')
print(f'  median: {statistics.median(word_counts):.0f}')
print(f'  max   : {max(word_counts)}')
print(f'  stdev : {statistics.stdev(word_counts):.0f}')

In [ ]:
# Sample example
sample = train[0]
print('=== Sample brief ===')
print(json.dumps(sample['brief'], indent=2, ensure_ascii=False))
print('\n=== Recommendation keys ===')
print(list(sample['recommendation'].keys()))